In [ ]:
!pip install transformers
!pip install  tensorflow
!wget "https://www.dropbox.com/scl/fo/vqig46u35fldpqdvyq54u/ANLgBoE4XEUIMsuuImgBEfs?rlkey=kqxps0ywhgnjz68h9503w4zm2&st=1roj6fus&dl=1" -O tuned_model.zip
!wget "https://www.dropbox.com/scl/fo/7yhpmkz5zdq9j4x4leopg/AOLsjkAIFcWEw7gI5WlqALg?rlkey=6v8u5a1qop0z1jurraqe212a1&st=t3di4z29&dl=0" -O wordbanks.zip


In [ ]:
import tensorflow as tf
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer
import torch
import numpy as np
import ast
import random
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import *
import zipfile

In [ ]:
with zipfile.ZipFile("tuned_model.zip", 'r') as zip_ref:
    zip_ref.extractall("tuned_model")
tuned_model = "tuned_model"
tokeniser = GPT2Tokenizer.from_pretrained(tuned_model)
model = TFGPT2LMHeadModel.from_pretrained(tuned_model, pad_token_id=tokeniser.eos_token_id)

with zipfile.ZipFile("wordbanks.zip", 'r') as zip_ref:
    zip_ref.extractall("wordbanks")

# **Fine-tuning and Conversion Code**

This code underneath was what I used to finetune the model, it downloads the Cornell Movie-Dialogs Corpus, it extracts akk the utterances, the actual speaking text and writes it into a new file and then trains the 355M gpt-2 model based on this. It will run for 5000 steps saving a checkpoint every 500 steps it completes. It's commented out so it's not run when run all is selected.

In [ ]:
'''
import gpt_2_simple as gpt2
from datetime import datetime
from convokit import Corpus, download

custom_model_dir = "/Documents/University Work/Postgrad/Comp Creative"

corpus = Corpus(filename=download("movie-corpus"))
utterances = [utt.text for utt in corpus.iter_utterances()]
with open("movie_corpus.txt", "w", encoding="utf-8") as f:
    for line in utterances:
        f.write(line + "\n")

sess = gpt2.start_tf_sess()
gpt2.finetune(sess,
              dataset="movie_corpus.txt",
              model_name='355M',
              model_dir=custom_model_dir,
              steps=5000,
              restore_from='latest',
              run_name='run_355M',
              overwrite = False,
              print_every=10,
              sample_every=100,
              save_every=500,
              accumulate_gradients=1
)
'''

This code underneath converts the TensorFlow checkpoint file to be usable with the HuggingFace functions.

In [ ]:
'''
# Load the GPT2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-medium')

# Load the GPT2 model architecture
model = TFGPT2LMHeadModel.from_pretrained('gpt2-medium', pad_token_id=tokenizer.eos_token_id)

# Load the fine-tuned checkpoint from gpt_2_simple
checkpoint_path = f"C:/Users/h4rry/checkpoint/run_355M/model-5011"

# Restore the weights from the fine-tuned checkpoint
checkpoint = tf.train.Checkpoint(model=model)
checkpoint.restore(checkpoint_path).expect_partial()

custom_model_dir = "C:/Users/h4rry/converted_model"
# Save the converted model to Hugging Face's format
model.save_pretrained(custom_model_dir)
tokenizer.save_pretrained(custom_model_dir)
'''

# **The Logit Influence Code**

# wordbank_processing
This is the code I used to filter through the NRC emotion wordbanks, that can be found here: https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm they are all seperated by emotion, each formatted as 'word' (1 or 0) if they have an affiliation with the word they will be encompanied by a 1 if not a 0, so it will go through and check which are relevant and which are not, then will create a capitalised verison of the word under the original, to increase their probabilities during generation.
**Parameters:**
* wordbank_path (str): users to path to local project folder where wordbanks are.
* fileheader (str): name of wordbank file.
* lines list<str> / <[str][str]>: this will either be a 1D list of a 2D list depending on input, it is set to None as a default value, when it is reading from a pre-existing wordbank file and this will contain the lines to write to a new wordbank file if this value is assigned.



In [ ]:
def wordbank_processing(wordbank_path, fileheader, lines = None):
  if(lines == None):
    file = open(str(wordbank_path) + str(fileheader) + '_wordbank.txt','r')
    lines = file.readlines()
    for i in range(len(lines)):
     lines[i] = lines[i].split()
     #print(lines[i][0])
    file.close();
  file = open(str(wordbank_path) + str(fileheader) + '_wordbank.txt','w')
  for i in range(len(lines)):
    line = lines[i]
    if isinstance(line, list) and len(line) > 1: #check to make sure it's a 1D list and not 2D list for the emotional wordbanks
        if(line[1] == '1'):
          file.write(line[0]+"\n")
          line_with_caps = list(line[0])
          line_with_caps[0] = line_with_caps[0].upper()
          line_with_caps = ''.join(line_with_caps)
          if(i+1 != len(lines)):
           if(lines[i+1] != line_with_caps):
             file.write(line_with_caps+"\n")
             i = i+2
    else:
       file.write(line+"\n")
       line_with_caps = list(line)
       line_with_caps[0] = line_with_caps[0].upper()
       line_with_caps = ''.join(line_with_caps)
       if(i+1 != len(lines)):
        if(lines[i+1] != line_with_caps):
          file.write(line_with_caps+"\n")
          i = i+2
  file.close()


# getting_wordlist
Loads a wordlist, tokenises words within, adds them all to list of all words in wordbank for processing in wordbank_p_increase_emotions

**Parameters:**
* emotion (str): current emotion associated with generated sentence.
* strpath (str): users to path to local project folder.
* multiple_token_words list<dict(int)(list<int>)> this is a list that holds dictionaries of all the words that are split into multiple tokens when tokenised, the keys for the dictionary is an int: the first token in the word that is tokenised, and the value is a list of tokens that follow that one. e.g. the words running and runnable are split into 2 tokens, in the dictionary run would be used as a key and the different word endings [ning] and [nable] would be the values.

**Returns:**
* wordbank_tokenised (list<float>): List of tokens for every word in wordbank
* multiple_token_words list<dict(int)(list<int>)> described above ^

In [ ]:
def getting_wordlist(topic, strpath, multiple_token_words):
    #print('getting_wordlist')
    file = str(strpath) + str(topic) + '_wordbank.txt'
    wordbank_tokenised = []
    '''
    This list just keeps track of all the words that correspond to multiple tokens,
    for example run, run by itself is just one token however there are variations
    like running and runnable these would count as seperate tokens, we need a
    way to store them in order to go back and complete these words if the first
    half is ever generated.
    '''
    multiple_token_words = {}
    '''
    Goes line by line through the text file and tokenises each word
    '''
    with open (file,'r') as data_file:
            for line in data_file:
                words = line.strip().split() #emotion wordbank contains vectors, we don't need to include those
                word_to_tokenise = words[0]
                word_tokenised = tokeniser(word_to_tokenise, return_tensors='np').input_ids
                wordbank_tokenised.append(int(word_tokenised[0][0]))

                if(len(word_tokenised[0]) > 1): #if the word is split into mutliple tokens
                  if(word_tokenised[0][0] in multiple_token_words): #this first token already exists
                      multiple_token_words[word_tokenised[0][0]].append(word_tokenised[0][1:].tolist()) #for words with more than 2 tokens, holds a list
                  else:
                      multiple_token_words[word_tokenised[0][0]] = [word_tokenised[0][1:].tolist()] #for words with more than 2 tokens, holds a list
    return wordbank_tokenised, multiple_token_words

# wordbank_p_increase
Increases the probabilities of the words within the given list of tokens in wordbank_tokenised by a predetermined amount

**Parameters:**
* word_probabilities (list<float>): List indexes of current probabilities of all words after transformation by Softmax function
* wordbank_tokenised (list<float>): List of tokens representative of topic or emotion wordbank
* p_increase (list<float>): Predetermined multiplier for the percentage increase for token apperance

**Returns:**
* new_word_probabilities (list<float>): List of probabilities, with probability increase to the indexes of the words within the wordlists

In [ ]:
def wordbank_p_increase(word_probabilities, wordbank_tokenised, p_increase):
    new_word_probabilities = word_probabilities.copy() #makes a copy of the original word_probabilities

# loop goes through all the words in the current word bank and increases their probability
    for token in wordbank_tokenised:
        new_word_probabilities[token] = new_word_probabilities[token] * p_increase
    return new_word_probabilities

# repetition_prevention
Check to see whether word that is selected already exists within the sequence, once list reaches threshold
it will pop word off the list, allowing for generated words to be reused

**Parameters:**
* repetition_counter (list<str>): a list of decoded words that the sequence has generated, the size of this list is determined by threshold.
* threshold (int): threshold value for max words in sequence before popping takes place.
* word_decoded list(str): this is the current most likely word attained by the top k search, this is used to check if it appears in the repetition_counter.
* check_counter (int): fail-safe value, if the function cannot find another word to generate that isn't in the list by this many iterations it will just generate the one with the highest probability

**Returns:**
* boolean: if True then generated word is accepted, if False new word must be generated

In [ ]:
def repetition_prevention(repetition_counter,word_decoded,threshold,check_counter):
    if(word_decoded not in repetition_counter):
        if (len(repetition_counter) >= threshold or check_counter > 40): #this can be improved
            repetition_counter.pop(0) #pops off the first item in the list, the oldest token in the list
            return True
        else:
            return True
    else:
        return False

# sample_from_top_k
Implementation of top-k search function, runs the worbank associated functions determined on the topic and emotion values, increasing the probabilities of the
words in those banks occuring, then collates the top k values according to the threshold with the highest probabilities then selects a value taking those probabilities
into account

**Parameters:**
* word_probabilities (list<float>): List of current probabilities of all words after transformation by Softmax function
*  repetition_counter (list<str>): current generated words in sequence
* threshold (int): threshold value for max words in sequence before popping takes place
* p_increase (list<float>): predetermined multiplier for the percentage
* topic (str): current topic associated with generated sentence
* emotion (str): current emotion associated with generated sentence
* banned_characters_tokens (list<float>): list of banned tokens to prevent creation of aberrant sentences, skipping these indexes in generation
* k (int): determines threshold value of generated word probabilities
* multiple_token_words list<dict(int)(list<int>)>: This is a list of dictionaries containing all words that contain multiple tokens
* wordbank_tokenised list<float>: A list of all the topic related words, tokenised
* emotion_wordbank_tokenised list<float>: A list of all the emotion related words, tokenised

**Returns:**
* most_likely_token_p_index (int): the index of the most likely next word in the sequence

In [ ]:
def sample_from_top_k(word_probabilities,repetition_counter,threshold,p_increase,topic,emotion, banned_characters_tokens, k, multiple_token_words,wordbank_tokenised,emotion_wordbank_tokenised):

    check_counter = 0
    incrimenter=0

# need to reshape the probabilites list to be able to extract indexes
    word_probabilities = np.reshape(word_probabilities, (1,-1))

# this converts the list of probabilites into a dictionary with the key being the index number and value being the probability
    word_indexes_prob = {}
    for i in range(word_probabilities[0].size):
      if i not in banned_characters_tokens: # prevents any indexes in banned_characters from being added
        word_indexes_prob[i] = word_probabilities[0][i]

# increasing the probabilities for the words in the wordbanks
    word_indexes_prob = wordbank_p_increase(word_indexes_prob, wordbank_tokenised, p_increase)
    word_indexes_prob = wordbank_p_increase(word_indexes_prob,emotion_wordbank_tokenised,p_increase)

# sorts these indexes by their values, probabilities, so only the top k values exist within the new list
    sorted_word_indices_prob = sorted(word_indexes_prob.items(), key=lambda x: x[1], reverse=True)
    word_indexes_prob.clear()
    for key, value in sorted_word_indices_prob:
        if(incrimenter < k):
            word_indexes_prob[key] = value
            incrimenter = incrimenter + 1

# this normalises the values of the probabilities by the sum of the probalities, now all between 0 and 1 or weighted selection won't work
    total_word_prob = sum(word_indexes_prob.values())
    for key in word_indexes_prob.keys():
        word_indexes_prob[key] = word_indexes_prob[key] / total_word_prob
        #if(key in wordbank_tokenised):
            #print("Word is in topic wordlist")
        #if(key in emotion_wordbank_tokenised):
            #print("Word is in emotion wordlist")
    total_word_prob = sum(word_indexes_prob.values())
    #print(word_indexes_prob)

    keys = list(word_indexes_prob.keys())
    probs = np.array([word_indexes_prob[k] for k in keys])
    probs = probs / np.sum(probs)

    if(len(word_indexes_prob.keys()) == 1):
        most_likely_token_k_index = next(iter(word_indexes_prob.keys())) #if there's only one option will just pick that
    else:
# Sample using the normalised probabilities,higher probability the higher chance it gets picked
        most_likely_token_k_index = np.random.choice(keys, p=probs)

        word_decoded = tokeniser.decode(most_likely_token_k_index, skip_special_tokens = True).replace(" ", "")
        check_bool = repetition_prevention(repetition_counter,word_decoded,threshold,check_counter)

# Performs a check to see if the token is in the repetition list, if it cannot find a valid token in so many iterations it will just pop off the token and choose that one
        while(check_bool == False ):
            check_counter += 1
            most_likely_token_k_index = np.random.choice(keys, p=probs)
            #most_likely_token_k_index = np.random.choice(list(word_indices_prob.keys()))
            word_decoded = tokeniser.decode(most_likely_token_k_index, skip_special_tokens = True).replace(" ", "")
            check_bool = repetition_prevention(repetition_counter,word_decoded,threshold,check_counter)
            #print("test is ",check_bool)
            #print("The word: ",word_decoded," The most likely index: ",most_likely_token_k_index)
            token_index = int(most_likely_token_k_index)
            if token_index in word_indexes_prob:
                 word_indexes_prob.pop(token_index)
            total_word_prob = sum(word_indexes_prob.values())
            for key in word_indexes_prob.keys():
                #print("Current value: ",value)
                word_indexes_prob[key] = word_indexes_prob[key] / total_word_prob

    return most_likely_token_k_index #Returns the most likely index

# generate_new_probabilities
This will generate a new set of probabilities for the system, based on the input system. It will take in the input system and get the models activations, it will extract specificaly on the logits. Once those are extracted we take the logits of the next predicted token specifically by reshaping the logits tuple. Divides by a temperature value in order to favour higher probability words. Softmax is applied to convert the logits into a probability distribution.

**Parameters:**
* input_sequence (list<str>): List of current probabilities of all words after transformation by Softmax function.
*  temperature (list<str>): current generated words in sequence.


**Returns:**
* probabilities list<int>: a list of probabilities, the probability distribution of tokens that may be the next token with the index of each item being the token number.

In [ ]:
def generate_new_probabilities(input_sequence,temperature):
    activations = model(input_sequence)[0] #extract specifically the logits
    logits = activations[:,-1,:] #logits reshaped to get logits on next predicted token only
    logits /= temperature
    probabilities = tf.nn.softmax(logits) #softmax is applied to convert into probability distribution
    return probabilities

# generate_sentence
This function handles the sentence and token evaluation evaluation and sentence concatination after tokens are selected using top k.

**Parameters:**
* current_NPC : the currently selected NPC.
* action : the currently selected action towards the NPC.
* max_length : the max number of tokens that are being generated.
* prob: predetermined multiplier for the percentage.
* generated_output: the dialogue box that is presented to the user after generation is complete.
* generation_progress : a progress bar that increases after each token is generated.
* file_path : filepath to retreive wordbanks.


In [ ]:
def generate_sentence(current_NPC,action,max_length,prob,generated_output,generation_progress):
  #initialising of variables
  generation_progress.max = max_length
  name = current_NPC.name
  topic = current_NPC.topic
  emotion = current_NPC.emotion
  users_action,input_sequence,output_sequence = action_to_sentence_starter(action,name,topic,emotion)
  generation_progress.value = 0
  generation_progress.style={'bar_color': 'red'}
  threshold = 20
  p_increase = prob
  strpath = "wordbanks/"
  sequence = tokeniser(input_sequence, return_tensors="np").input_ids
  output_sequence = tokeniser(output_sequence, return_tensors="np").input_ids
  current_input = tokeniser(input_sequence, return_tensors="np").input_ids
  layer = model.get_layer("transformer")
  repetition_counter = []
  #These were special characters that were causing particular difficulties during generation
  banned_characters = ['[',' [',']','~','(',')','"','_','*','你','�','ㅋ','」',';',':','<','>','#','://']
  banned_characters_tokens = []
  #goes through banned words list and tokenises them
  for character in banned_characters:
      token_ids = tokeniser(character, return_tensors="np").input_ids
      banned_characters_tokens.append(int(token_ids[0][0]))  # extract integer token ID
  temperature = 1.5
  max_k = 50
  multiple_token_words = {}

#tokenises the topic wordbank
  wordlist_output = getting_wordlist(topic,strpath,multiple_token_words)
  wordbank_tokenised = wordlist_output[0]
  multiple_token_words |= wordlist_output[1]

#tokenises the emotion wordbank
  wordlist_output = getting_wordlist(emotion,strpath,multiple_token_words)
  emotion_wordbank_tokenised = wordlist_output[0]
  multiple_token_words |= wordlist_output[1]

#loop will run until it hit max sentence length
  i=0
  while i <(max_length):
      # the sequence has it's probabilities generated and then top k is sampled
      probabilities = generate_new_probabilities(sequence,temperature)
      most_likely_token_k_index = sample_from_top_k(probabilities,repetition_counter,threshold,p_increase
                                                  ,topic,emotion,banned_characters_tokens,max_k,multiple_token_words
                                                  ,wordbank_tokenised,emotion_wordbank_tokenised)

      most_likely_token_k_index = np.reshape(most_likely_token_k_index, (1,1))
      most_likely_token = tokeniser.decode(most_likely_token_k_index[0], skip_special_tokens = True)
      #checking for banned characters and some additional tricky tokens that were causing errors
      check = [char for char in banned_characters if(char in most_likely_token)]
      if(check or most_likely_token_k_index == 198 or most_likely_token_k_index == 628 or most_likely_token_k_index == 1849 or most_likely_token_k_index == 10221):
        continue
      else:
            #if the token is successfully chosen it will check to see if token is a full word or a word split into multiple tokens
            token_check = int(most_likely_token_k_index[0][0])
            if token_check in multiple_token_words:
              # Multiple token word found so the system will decode this and add a space in front of it for proper
              # sentence structure and this is then concatinated onto the sequence
              token_decoded = tokeniser.decode(most_likely_token_k_index[0], skip_special_tokens = True).replace(" ", "")
              token_with_space = tokeniser(" "+token_decoded, return_tensors='np').input_ids
              repetition_counter.append(token_decoded.replace(" ", ""))
              generation_progress.value += 1  #progress bar update
              sequence = tf.concat([sequence, token_with_space], axis=1)
              output_sequence = tf.concat([output_sequence, most_likely_token_k_index], axis=1)
              i+=1
              current_text = tokeniser.decode(current_input[0])
              current_input = tokeniser(current_text, return_tensors="np").input_ids
              # a fresh set of probabilities is generated for the sequence with the new last token added to it in order to determine which word ending is most appropriate for the sentence
              probabilities = generate_new_probabilities(current_input,temperature)
              next_token_list = []

              for continuation in multiple_token_words[token_check]:
                next_token_list.append(continuation[0])
              # picks the one with the highest probability, this is unaffected by the probability increase, so no bias, just the one that makes the most sense for the already generated text
                best_next_token = max(next_token_list, key=lambda t: probabilities[0][t])

              # finds this highest next token and gets the full multi-token continuation from it
              for continuation in multiple_token_words[token_check]:
                if(best_next_token == continuation[0]):
                  chosen_continuation = continuation
                  break

              # adds the rest of the tokens onto the sequence
              for token in chosen_continuation:
                most_likely_token_k_index = np.reshape(token, (1,1))
                most_likely_token = tokeniser.decode(most_likely_token_k_index[0], skip_special_tokens = True)
                sequence = tf.concat([sequence, most_likely_token_k_index], axis=1)
                output_sequence = tf.concat([output_sequence, most_likely_token_k_index], axis=1)
                #print((tokeniser.decode(sequence[0])))
              i+=1
            else:
              # if it's not a multi-token word it will just add it to the repetition counter and add it to the sequence
              repetition_counter.append(most_likely_token.replace(" ", ""))
              sequence = tf.concat([sequence, most_likely_token_k_index], axis=1)
              output_sequence = tf.concat([output_sequence, most_likely_token_k_index], axis=1)
              generation_progress.value += 1 #progress bar update
              i+=1

#once generation has finished it will look for the lastest instance of a sentence ending and cut off anything further than that and will then output that result to the user
  final_sentence = (tokeniser.decode(output_sequence[0]))
  sentence_endings = ['.', '!', '?']
  last_index = max(final_sentence.rfind(punct) for punct in sentence_endings)
  final_sentence = final_sentence[:last_index + 1]
  current_generated_output = generated_output.value
  inner_content = current_generated_output.replace('<div style="height:300px; overflow:auto;">', '').replace('</div>', '')
  inner_content += f"{users_action}<br>{final_sentence}<br>"
  generated_output.value = f'<div style="height:200px; overflow:auto;">{inner_content}</div>'
  generation_progress.style = {'bar_color': 'green'}


#NPC Class
Class for NPC objects, each holds their own name, emotion and topic.

In [ ]:
class NPC:
  def __init__(self, name, emotion, topic):
    self.name = name
    self.emotion = emotion;
    self.topic = topic;
NPC_List = []
NPC_List.append(NPC("Lola","angry","flowers"))
NPC_List.append(NPC("Harry","happy","flowers"))
NPC_List.append(NPC("George","sad","flowers"))
emotion_list = ['angry','disgust','fear','joy','sadness']
topic_list = ['flowers']


# get_NPC_names

A getter method for the names of all the NPC's in the list

**Returns:**
* NPC_names_list: Returns the names of all the NPCS


In [ ]:
def get_NPC_names():
  NPC_names_list = []
  for npc in NPC_List:
    NPC_names_list.append(npc.name)
  return NPC_names_list

# get_topics

A getter method for the topics

**Returns:**
* topic_list List(str): Returns all the topics


In [ ]:
def get_topics():
  return topic_list

# get_emotions

A getter method for the emotions

**Returns:**
* emotion_list List(str): Returns all the emotions

In [ ]:
def get_emotions():
  return emotion_list

# create_new_npc

Function that creates and adds an new NPC to the NPC list

**Parameters:**
* name (str): The name of the NPC.
* emotion (str): The NPCs emotion.
* topic (str): The NPCs topic.

**Returns:**
* bool: This is used to verify whether the new NPC entry is valid or not, if it is not it will return false to the user an dwill be displayed on the UI, if it is it will be added to the NPC
list and displayed as successful to the user.


In [ ]:
def create_new_npc(name, emotion, topic):
  if(name == ""): #case for empty input field
    return False
  for npc in NPC_List:
    if npc.name == name: #case for name already existing in the list
      return False
  for letter in name:
    if(letter.isdigit()): #case for if name has numbers in it
      return False

  NPC_List.append(NPC(name,emotion,topic))
  #print(NPC_List)
  return True

# **User Interface Related Code**

# change_Speaking_NPC
Takes an NPC's name as an input and then searches through the NPC List to find the NPC named and returns the currently selected npc.

 **Parameters:**
* name (str): The name of the NPC that the user wants to switch to.
* NPC_List List(NPC): A list of NPC objects, currently in the system.

**Returns:**
* currently_selected_NPC (NPC): Returns the NPC with the name value that matches the one given in the parameter.

In [ ]:
def change_Speaking_NPC(name, NPC_List):
  for npc in NPC_List:
    if npc.name == name:
      currently_selected_NPC = npc
  return currently_selected_NPC

# change_NPC_attributes
Takes an NPC's name as an input and then searches through the NPC List to find the NPC named and changes their topic and emotion based on what the user selected.

 **Parameters:**
* name (str): The name of the NPC that the user wants to switch to.
* NPC_List List(NPC): A list of NPC objects, currently in the system.
* emotion (str): This is the new emotion that the NPC will have set
* topic (str): This is the new topic that the NPC will have set
* change_npc_outcome (widget.HTML): This will just display the NPC changed feedback for the user no need to change this as all of the options are dropdowns.



In [ ]:
def change_NPC_attributes(b,name, NPC_List,emotion,topic,change_npc_outcome):
  for npc in NPC_List:
    if npc.name == name:
      npc.emotion = emotion
      npc.topic = topic
      change_npc_outcome.layout.display = 'block'

# switch_UI
This function simply displays one of the UI pages and hides the other, depending on which UI page is open at function call.

 **Parameters:**
* b (str): placeholder for button click event object.
* StartScreen_UI HBox[(other UI widgets)]: Just a box containing all the UI elements displayed on the start screen
* CreateScreen_UI HBox[(other UI widgets)]: Just a box containing all the UI elements displayed on the create screen

In [ ]:
def switch_UI(b,StartScreen_UI,CreateScreen_UI):
    if(StartScreen_UI.layout.display == 'none'): #Start screen is not being displayed
        StartScreen_UI.layout.display = 'flex'
        CreateScreen_UI.layout.display = 'none'
    else:  #Create screen is not being displayed
        StartScreen_UI.layout.display = 'none'
        CreateScreen_UI.layout.display = 'flex'

# switch_NPC
This function allows the current NPC to be changed based on the button clicked on the NPC grid, this will not only update the local variable that holds the value, but also update the UI elements.

 **Parameters:**
* change {}: Dictionary used by widgets, it contains information about change events corresponding to button presses, examples used in my code 'new' = a button has been pressed and 'owner' = the specific button that has been pressed.
* currently_selected_NPC (NPC): A list of NPC objects, currently in the system.
* name_label (widgets.HTML): The UI element that displays the currently selected NPCs name.
* emotion_label (widgets.HTML): The UI element that displays the currently selected NPCs emotion.
* topic_label (widgets.HTML): The UI element that displays the currently selected NPCs topic.
* buttons List(widgets.button): A list of the NPC buttons in the button grid, these will be clicked to select an NPC and rest will be turned off.


In [ ]:
def switch_NPC(change,currently_selected_NPC,name_label,emotion_label,topic_label,buttons,generated_output):
    if change['new']: #if button is clicked
      if change['owner'].description != ' ': #if the current NPC isn't blank
        currently_selected_NPC['npc'] = change_Speaking_NPC(change['owner'].description,NPC_List) #update the currently selected NPC and the UI elements
        name_label.value = (f"<div style='font-size:20px; font-weight:bold;'>Name: {currently_selected_NPC.get('npc').name} </div>")
        emotion_label.value = (f"<div style='font-size:20px; font-weight:bold;'>Emotion: {currently_selected_NPC.get('npc').emotion}</div>")
        topic_label.value = (f"<div style='font-size:20px; font-weight:bold;'>Topic: {currently_selected_NPC.get('npc').topic} </div>")
        generated_output.value = ''
        for btn in buttons:
            if btn != change['owner']: # button is not the one that's clicked
                btn.value = False # turn off button
      else:
        change['owner'].value = False # turn off button

# get_current_npc
Getter method for attainting the currently selected npc.

 **Parameters:**
* NPC_List List(NPC): List of all NPCs
* name_label (widget.html): The label that is currently displaying the name of the npc, using this because it is changed at runtime and as all other values seem to be preset this is the only way I could seem to retrieve it

In [ ]:
def get_current_npc(NPC_List, name_label):
    displayed_name = name_label.value.split("Name: ")[1].split(" </div>")[0] #gets value from name_label as is updated during runtime
    for npc in NPC_List:
        if npc.name == displayed_name:
            return npc

# generate_npc_buttons
This method is used to generate all the buttons in the grid, it will take the NPC names and create a button for each of them also assigning the switch_NPC function to them onclick.

 **Parameters:**
* currently_selected_NPC List(NPC): The currently selected NPC
* name_label (widget.html): The label that is currently displaying the name of the npc to pass into the switchNPC function
* emotion_label (widget.html): The label that is currently displaying the emotion of the npc to pass into the switchNPC function
* topic_label (widget.html): The label that is currently displaying the topic of the npc to pass into the switchNPC function
* NPC_names List(str): A list of all the currently created NPCs names

In [ ]:
def generate_npc_buttons(currently_selected_NPC,name_label,emotion_label,topic_label,NPC_names,generated_output):
  buttons = []
  for name in NPC_names: #creates new buttons for each NPC
    button = widgets.ToggleButton(description=name)
    button.layout.height = '145px'
    button.layout.width = '175px'
    buttons.append(button)

  for button in buttons: # assigns obverse to each button to call function when clicked
    button.observe(lambda change: switch_NPC(change,currently_selected_NPC,name_label,emotion_label,topic_label,buttons,generated_output), names='value')

# sets up the structure of the NPC box grid
  left_column = HBox(buttons[0:3])
  middle_column= HBox(buttons[3:6])
  right_column= HBox(buttons[6:9])
  NPC_Buttons = VBox([left_column, middle_column, right_column])
  return NPC_Buttons

# on_create_npc_button_clicked
Function assigned to the create npc button, this will perform a call the function create_new_npc this will return a bool if the bool is true it will give feedback to the user that their action was successful if not it will provide feedback to enter a correct or new name.


In [ ]:
def on_create_npc_button_clicked(b,npc_name_field,npc_emotion_dropdown,npc_topic_dropdown,create_name_outcome):
    npc_created_bool = create_new_npc(npc_name_field.value, npc_emotion_dropdown.value, npc_topic_dropdown.value)
    if npc_created_bool: #if the npc has successfully been created
        create_name_outcome.value = "<div style='font-size:15px; color:green;'>NPC created successfully </div>"
        create_name_outcome.layout.display = 'block'
    else: #if the npc has not successfully been created
        create_name_outcome.value = "<div style='font-size:15px; color:red;'>Invalid NPC name, please retry </div>"
        create_name_outcome.layout.display = 'block'

# on_create_wordbank_button_clicked
Function that defines the action performed when the button to create a worbank is pressed, first it performs a check to see whether both fields have a valid input, if they do it will take the inputs from both fields and write a new wordbank into the users chosen directory so that it can be used for text generation.



In [ ]:
def on_create_wordbank_button_clicked(create_wordbank_field,new_wordbank_area,create_wordbank_outcome,file_path):
    #if there is a blank input in the text field or the input already exists within the topic list or it is the same as an emotion
    if(create_wordbank_field.value == "" or create_wordbank_field.value in topic_list or create_wordbank_field.value in emotion_list):
      create_wordbank_outcome.value = "<div style='font-size:15px; color:red;'>Invalid wordbank name, please retry </div>"
      create_wordbank_outcome.layout.display = 'block'
      return False
    #this is the check for the wordbank text area, there must be atleast 20 entries for it to be accepted.
    wordbank_input_words = new_wordbank_area.value.split('\n')
    if(len(wordbank_input_words) < 20):
      create_wordbank_outcome.value = "<div style='font-size:15px; color:red;'>Invalid wordbank entry, please retry </div>"
      create_wordbank_outcome.layout.display = 'block'
      return False
    #If both checks are valid a new file will be created by the wordbank_processing function and the topic will be added to the list
    wordbank_processing(file_path,create_wordbank_field.value,wordbank_input_words)
    topic_list.append(create_wordbank_field.value)
    create_wordbank_outcome.value = "<div style='font-size:15px; color:green;'>Wordbank created successfully </div>"
    create_wordbank_outcome.layout.display = 'block'
    return True

# action_to_sentence_starter
This function takes in the action that user has selected to take against the NPC, the NPCs name, their selected topic and their emotion and returns the sentence prompt that will be used to start generation.


In [ ]:
def action_to_sentence_starter(action,name,topic,emotion):
    random = np.random.randint(1,4)
    match action:
      case "Insult Topic":
        users_action = "You insulted " + name + "'s " + topic + "."
        npc_response = "[" + emotion + "] " + name + ":"
        return users_action,str(users_action+" "+npc_response),npc_response
      case "Neutral Greeting":
        users_action = "You greeted " + name + "."
        npc_response = "[" + emotion + "] " + name + ":"
        return users_action,str(users_action+" "+npc_response),npc_response
      case "Compliment Topic":
        users_action = "You complimented " + name + "'s " + topic + "."
        npc_response = "[" + emotion + "] " + name + ":"
        return users_action,str(users_action+" "+npc_response),npc_response
      case "Ask about Topic":
        users_action = "You asked " + name + " about " + topic + "."
        npc_response = "[" + emotion + "] " + name + ":"
        return users_action,str(users_action+" "+npc_response),npc_response


# create_ui
Although long this function essentially just creates the UI for the system, I have subdivided it into sections with comments.



In [ ]:
def create_ui(NPC_List,emotion_list,topic_list,action_list):
  file_path = "wordbanks/"
  currently_selected_NPC = {'npc': NPC_List[0]} #Needs to be a dictionary in order to update correctly

#The right-side (generation tab) of the Start Screen
  name_label = widgets.HTML(f"<div style='font-size:20px; font-weight:bold;'>Name: {currently_selected_NPC['npc'].name} </div>")
  emotion_label = widgets.HTML(f"<div style='font-size:20px; font-weight:bold;'>Emotion: {currently_selected_NPC['npc'].emotion} </div>")
  topic_label = widgets.HTML(f"<div style='font-size:20px; font-weight:bold;'>Topic: {currently_selected_NPC['npc'].topic} </div>")

  #initalises the NPC names for use in the buttons
  NPC_names = get_NPC_names()
  if(len(NPC_names) < 9):
    for i in range(9-len(NPC_names)):
      NPC_names.append(" ")

  #generated output box
  generated_output = widgets.HTML(
    value='<div style="height:300px; overflow:auto;"></div>',
    placeholder='Output will appear here',
    layout=widgets.Layout(height='300px', width='500px', border='1px solid gray', padding='10px')
  )

  #slider for probability increase set from 0 to 0.25 as anything larger cause large number errors where the probabilities get too large
  word_intensity_slider =widgets.FloatSlider(
    value=0.15,
    min=0.0,
    max=0.25,
    step=0.01,
    description='Wordbank Influence Intensity:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
    layout=widgets.Layout(width='100%')
  )
  word_intensity_slider.style.description_width = '175px'

  #Input field for desired text length
  word_length_input = widgets.BoundedIntText(
    value=35,
    min=10,
    max=100,
    step=1,
    description='Desired Text Length:',
    disabled=False
 )
  word_length_input.style.description_width = '140px'

  #dropdown for the actions
  action_dropdown = widgets.Dropdown(
    value = action_list[1],
    options= action_list,
    description='Choose an action:',
    layout=widgets.Layout(width='75%')
  )
  action_dropdown.style.description_width = '140px'

  #generate sentence button
  generate_sentence_button = widgets.Button(description="Generate Sentence")
  centered_button = widgets.HBox([generate_sentence_button], layout=widgets.Layout(justify_content='center'))
  p_increase = 1 + word_intensity_slider.value

  #calls generate sentence function when button is clicked passing in NPC details
  generate_sentence_button.on_click(lambda b: (
    #print(currently_selected_NPC['npc'].name),
    generate_sentence(currently_selected_NPC['npc'],action_dropdown.value,word_length_input.value,(1 + word_intensity_slider.value),generated_output,generation_progress)
    ))

  #progress bar to show the user when their dialouge is generated
  generation_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=word_length_input.value,
    description='Generating Response:',
    bar_style='',
    style={'bar_color': 'red'},
    orientation='horizontal',
    layout=widgets.Layout(width='75%')
  )
  generation_progress.style.description_width = '140px'
  centered_progress = widgets.HBox([generation_progress], layout=widgets.Layout(justify_content='center'))

  #Box to hold all the elements at the right side of the screen
  NPC_Display = VBox([name_label,emotion_label,topic_label,generated_output,word_intensity_slider,word_length_input,action_dropdown,centered_button,generation_progress])

#The left-side (npc selection) of the Start Screen
  #generates the grid of npc buttons
  NPC_Buttons = generate_npc_buttons(currently_selected_NPC,name_label,emotion_label,topic_label,NPC_names,generated_output)
  NPC_Buttons_Box = VBox([NPC_Buttons])

  #label for the grid
  grid_label = widgets.HTML("<div style='font-size:20px; font-weight:bold;'>Pick an NPC: </div>")

  # create screen button
  create_screen_button = widgets.Button(description="Create Screen")

  #encapsulates the grid, labels and buttons
  NPC_Button_Grid = VBox([grid_label,NPC_Buttons_Box,create_screen_button])

  #encapsulates all the ui elements on the start screen
  StartScreen_UI = HBox([NPC_Button_Grid,NPC_Display],
                      layout=widgets.Layout(width='1100px', height='600px', border='solid 1px #ccc', padding='10px'))

  # assigns the switch UI function to the create npc button to switch screens
  create_screen_button.on_click(lambda b :switch_UI(b,StartScreen_UI,CreateScreen_UI))

#The left-side (npc change and creation) of the creation screen
  #label for the create npc box
  create_npc_label = widgets.HTML("<div style='font-size:20px; font-weight:bold;'>Create an NPC: </div>")
  #npc name field to type new npc name
  npc_name_field = widgets.Text(
    value='',
    placeholder='Enter NPC name',
    description='NPC Name:',
    disabled=False
  )
  #dropdown to select an emotion
  npc_emotion_dropdown = widgets.Dropdown(
    options= get_emotions(),
    description='NPC Starting Emotion:',
  )
  npc_emotion_dropdown.style.description_width = '140px'
  #dropdown to select a topic
  npc_topic_dropdown = widgets.Dropdown(
    options= get_topics(),
    description='NPC Topic:'
  )

  #feedback label for the user to tell them whether their npc was created or not
  create_name_outcome = widgets.HTML("<div style='font-size:15px; color:red;'>Invalid NPC name, please retry </div>")
  create_name_outcome.layout.display = 'none'
  #create npc button, is assigned to the on_create_npc_button_clicked function, just creates a new npc with the inputs given in the field and dropdowns
  create_npc_button = widgets.Button(description="Create NPC")
  centered_npc_button = widgets.HBox([create_npc_button], layout=widgets.Layout(justify_content='center'))

  create_npc_button.on_click(lambda b: (
    on_create_npc_button_clicked(b, npc_name_field, npc_emotion_dropdown, npc_topic_dropdown, create_name_outcome)
))

  #label for the change npc box
  change_npc_label = widgets.HTML("<div style='font-size:20px; font-weight:bold;'>Change an NPCs emotion and Topic: </div>")
  #dropdown to select an npc
  npc_name_dropdown = widgets.Dropdown(
      options= get_NPC_names(),
    description='NPC Names:',
  )
  #dropdown to select an emotion
  change_emotion_dropdown = widgets.Dropdown(
    options= topic_list,
    description='NPC Starting Emotion:',
  )
  change_emotion_dropdown.style.description_width = '140px'
  #dropdown to select a topic
  change_topic_dropdown = widgets.Dropdown(
    description='NPC Topic:'
  )
  change_topic_dropdown.options= emotion_list

  #feedback label for the user to tell them their npcs attributes were changed
  change_npc_outcome = widgets.HTML("<div style='font-size:15px; color:green;'>NPC successfully changed </div>")
  change_npc_outcome.layout.display = 'none' #hides the label at start

  #change npc button, is assigned to the change_NPC_attributes function, just changes the npc with the inputs given in the dropdowns
  change_npc_button = widgets.Button(description="Change NPC")
  #centers the button
  centered_change_button = widgets.HBox([change_npc_button], layout=widgets.Layout(justify_content='center'))

  change_npc_button.on_click(lambda b: (
    change_NPC_attributes(b,npc_name_dropdown.value,NPC_List,change_emotion_dropdown.value,change_topic_dropdown.value,change_npc_outcome)
))


  #back button simiarly to the create screen button switches between the two different screens
  back_button = widgets.Button(description="Back")
  bottom_left_box = widgets.Box([back_button],
    layout=widgets.Layout(
        justify_content='flex-start',
        align_items='flex-end',
        height='90%',
    )
)

  back_button.on_click(lambda b: switch_UI(b,StartScreen_UI,CreateScreen_UI))

  #this box encapsulates both parts of the left side of the screen
  create_npc = VBox([create_npc_label,npc_name_field,npc_emotion_dropdown,npc_topic_dropdown,create_name_outcome,centered_npc_button])
  change_npc = VBox([change_npc_label,npc_name_dropdown,change_emotion_dropdown,change_topic_dropdown,change_npc_outcome,centered_change_button])

  #this box encapsulates the whole left side of the screen
  npc_edit_tab = VBox([create_npc,change_npc,back_button])

#The right-side (wordbank creation) of the creation screen
  #create wordbank label
  create_wordbank_label = widgets.HTML("<div style='font-size:20px; font-weight:bold;'>Create a Wordbank: </div>")
  #create wordbank field for inputting the wordbanks name
  create_wordbank_field = widgets.Text(
    placeholder='Enter wordbank name',
    description='Wordbank Name:',
    disabled=False,
    layout=widgets.Layout(width='50%')
  )
  create_wordbank_field.style.description_width = '125px'
  #this is just a little note telling the user they must input 20 words for the bank
  create_wordbank_note= widgets.HTML("<div style='font-size:15px;'>Use the box below to define your wordbank, it must contain atleast 20 words, 1 word per line. </div>")

  # this is the area where they input the words for the wordbank
  new_wordbank_area = widgets.Textarea(
    placeholder='Enter wordbank entries',
    description='String:',
    disabled=False,
    layout=widgets.Layout(width='600px', height='300px')
)

  # a label to give feedback to the user on whether their wordbank creation was successful or not
  create_wordbank_outcome = widgets.HTML("<div style='font-size:15px; color:red;'>Invalid wordbank entry, please retry </div>")
  create_wordbank_outcome.layout.display = 'none' #hides it on start
  # this is the button that is pressed when the user has input all their words for the bank it calls the on_create_wordbank_button_clicked function
  create_wordbank_button = widgets.Button(description="Create Wordbank")
  centered_wordbank_button = widgets.HBox([create_wordbank_button], layout=widgets.Layout(justify_content='center'))


  create_wordbank_button.on_click(lambda b: (
    on_create_wordbank_button_clicked(create_wordbank_field,new_wordbank_area,create_wordbank_outcome,file_path)
    ))

  #encapsulates the right side of the screen
  create_wordbank = VBox([create_wordbank_label,create_wordbank_field,create_wordbank_note,new_wordbank_area,create_wordbank_outcome,centered_wordbank_button],
                         Layout=widgets.Layout(border='solid 50px'))

  #encapsulates the whole of the create screen
  CreateScreen_UI = HBox([npc_edit_tab,create_wordbank],
                      layout=widgets.Layout(width='1100px', height='600px', border='solid 1px #ccc', padding='10px'))

  #sets the start screen to display and the create screen to hide when run and assigns the whole UI to display when function is called
  CreateScreen_UI.layout.display = 'none'
  StartScreen_UI.layout.display = 'flex'
  display(VBox([StartScreen_UI,CreateScreen_UI]))

# Default Values

In [ ]:
NPC_List = []
NPC_List.append(NPC("Lola","anger","flowers"))
NPC_List.append(NPC("Jacob","joy","music"))
NPC_List.append(NPC("Harry","sadness","sports"))
NPC_List.append(NPC("Katy","disgust","fantasy"))
NPC_List.append(NPC("Nathanael","fear","technology"))
topic_list= ['flowers','music','sports','fantasy','technology']
emotion_list = ['anger','disgust','fear','joy','sadness']
action_list = ['Insult Topic','Neutral Greeting','Compliment Topic','Ask about Topic']

# **This is the cell to run to run programme ↓**
Note: if you create an NPC or add a wordbank, refresh this cell for some reason they don't update at runtime

In [ ]:
create_ui(NPC_List,topic_list,emotion_list,action_list)

# Evaluatory Functions

This is just a standalone version of the main generate sentence script that doesn't need the GUI to use

In [ ]:
def generate_sentence_standalone(current_NPC,random_topic,random_emotion,action,max_length,prob):
  #initialising of variables
  name = current_NPC.name
  topic = random_topic
  emotion = random_emotion
  users_action,input_sequence,output_sequence = action_to_sentence_starter(action,name,topic,emotion)
  threshold = 20
  p_increase = prob
  strpath = "wordbanks/"
  sequence = tokeniser(input_sequence, return_tensors="np").input_ids
  output_sequence = tokeniser(output_sequence, return_tensors="np").input_ids
  current_input = tokeniser(input_sequence, return_tensors="np").input_ids
  layer = model.get_layer("transformer")
  repetition_counter = []
  #These were special characters that were causing particular difficulties during generation
  banned_characters = ['[',' [',']','~','(',')','"','_','*','你','�','ㅋ','」',';',':','<','>','#','://']
  banned_characters_tokens = []
  #goes through banned words list and tokenises them
  for character in banned_characters:
      token_ids = tokeniser(character, return_tensors="np").input_ids
      banned_characters_tokens.append(int(token_ids[0][0]))  # extract integer token ID
  temperature = 1.5
  max_k = 50
  multiple_token_words = {}

#tokenises the topic wordbank
  wordlist_output = getting_wordlist(topic,strpath,multiple_token_words)
  wordbank_tokenised = wordlist_output[0]
  multiple_token_words |= wordlist_output[1]

#tokenises the emotion wordbank
  wordlist_output = getting_wordlist(emotion,strpath,multiple_token_words)
  emotion_wordbank_tokenised = wordlist_output[0]
  multiple_token_words |= wordlist_output[1]

#loop will run until it hit max sentence length
  i=0
  while i <(max_length):
      # the sequence has it's probabilities generated and then top k is sampled
      probabilities = generate_new_probabilities(sequence,temperature)
      most_likely_token_k_index = sample_from_top_k(probabilities,repetition_counter,threshold,p_increase
                                                  ,topic,emotion,banned_characters_tokens,max_k,multiple_token_words
                                                  ,wordbank_tokenised,emotion_wordbank_tokenised)

      most_likely_token_k_index = np.reshape(most_likely_token_k_index, (1,1))
      most_likely_token = tokeniser.decode(most_likely_token_k_index[0], skip_special_tokens = True)
      #checking for banned characters and some additional tricky tokens that were causing errors
      check = [char for char in banned_characters if(char in most_likely_token)]
      if(check or most_likely_token_k_index == 198 or most_likely_token_k_index == 628 or most_likely_token_k_index == 1849 or most_likely_token_k_index == 10221):
        continue
      else:
            #if the token is successfully chosen it will check to see if token is a full word or a word split into multiple tokens
            token_check = int(most_likely_token_k_index[0][0])
            if token_check in multiple_token_words:
              # Multiple token word found so the system will decode this and add a space in front of it for proper
              # sentence structure and this is then concatinated onto the sequence
              token_decoded = tokeniser.decode(most_likely_token_k_index[0], skip_special_tokens = True).replace(" ", "")
              token_with_space = tokeniser(" "+token_decoded, return_tensors='np').input_ids
              repetition_counter.append(token_decoded.replace(" ", ""))
              sequence = tf.concat([sequence, token_with_space], axis=1)
              output_sequence = tf.concat([output_sequence, most_likely_token_k_index], axis=1)
              i+=1
              current_text = tokeniser.decode(current_input[0])
              current_input = tokeniser(current_text, return_tensors="np").input_ids
              # a fresh set of probabilities is generated for the sequence with the new last token added to it in order to determine which word ending is most appropriate for the sentence
              probabilities = generate_new_probabilities(current_input,temperature)
              next_token_list = []

              for continuation in multiple_token_words[token_check]:
                next_token_list.append(continuation[0])
              # picks the one with the highest probability, this is unaffected by the probability increase, so no bias, just the one that makes the most sense for the already generated text
                best_next_token = max(next_token_list, key=lambda t: probabilities[0][t])

              # finds this highest next token and gets the full multi-token continuation from it
              for continuation in multiple_token_words[token_check]:
                if(best_next_token == continuation[0]):
                  chosen_continuation = continuation
                  break

              # adds the rest of the tokens onto the sequence
              for token in chosen_continuation:
                most_likely_token_k_index = np.reshape(token, (1,1))
                most_likely_token = tokeniser.decode(most_likely_token_k_index[0], skip_special_tokens = True)
                sequence = tf.concat([sequence, most_likely_token_k_index], axis=1)
                output_sequence = tf.concat([output_sequence, most_likely_token_k_index], axis=1)
                #print((tokeniser.decode(sequence[0])))
              i+=1
            else:
              # if it's not a multi-token word it will just add it to the repetition counter and add it to the sequence
              repetition_counter.append(most_likely_token.replace(" ", ""))
              sequence = tf.concat([sequence, most_likely_token_k_index], axis=1)
              output_sequence = tf.concat([output_sequence, most_likely_token_k_index], axis=1)
              i+=1

#once generation has finished it will look for the lastest instance of a sentence ending and cut off anything further than that and will then output that result to the user
  final_sentence = (tokeniser.decode(sequence[0]))
  sentence_endings = ['.', '!', '?']
  last_index = max(final_sentence.rfind(punct) for punct in sentence_endings)
  final_sentence = final_sentence[:last_index + 1]
  return final_sentence


This is the code I used to randomly generate outputs commented out so it dosen't run

In [ ]:
'''
finetuned_and_prob_adjusted_list = []

for _ in range(30):
    random_npc = random.choice(NPC_List)
    random_topic = random.choice(topic_list)
    random_emotion = random.choice(emotion_list)
    random_action = random.choice(action_list)
    random_length = random.randint(20, 100)
    prob = 1.14

    sentence = generate_sentence_standalone(
        random_npc, random_topic, random_emotion, random_action, random_length, prob
    )

    finetuned_and_prob_adjusted_list.append(sentence)

with open("generated_sentences2.txt", "w", encoding="utf-8") as file:
    for idx, sentence in enumerate(finetuned_and_prob_adjusted_list, 1):
        file.write(f"{idx}. {sentence}\n\n")

print("Sentences written to 'generated_sentences.txt'")
'''

# Emotional Analysis Code
This is the code for the emotional analysis, it creates a regular expression to extract the emotion tag from the sentence so it dosen't influence calssifying. The goes through all the sentences and classifys them using the Emotion English DistilRoBERTa-base then checks whether the emotion it was classified as matches the one it was expected to be. Again commented to stop from running.

In [ ]:
'''
import re
from transformers import pipeline

# Load the emotion analysis model
emotion_analyzer = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=False
)

results = []

# Read and analyze each line
with open("generated_sentences2.txt", "r", encoding="utf-8") as file:
    lines = [line.strip() for line in file if line.strip()]

    for idx, line in enumerate(lines, 1):
        cleaned_text, original_tag = extract_emotion_and_clean_text(line)
        if not cleaned_text:
            continue

        # Run emotion analysis
        emotion = emotion_analyzer(cleaned_text)[0]

        # Check if original tag matches detected emotion
        matches = original_tag.lower() == emotion['label'].lower()

        results.append({
            'idx': idx,
            'text': cleaned_text,
            'original_tag': original_tag,
            'detected_emotion': emotion['label'],
            'confidence': emotion['score'],
            'matches': matches
        })
'''